In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [2]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [3]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
import os
from dotenv import load_dotenv
from google import genai
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()

client = genai.Client(
api_key=os.getenv("GOOGLE_API_KEY")
)


In [6]:
import pandas as pd

df_answers = pd.read_csv("../data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [7]:
rec = answers[0]
rec

{'question': "Is it possible to enroll in the course even if I've just found it?",
 'answer_llm': "Yes, you can still join the course even if you've just discovered it. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.",
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [8]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Is it possible to enroll in the course even if I've just found it?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join the course even if you've just discovered it. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.


In [10]:
eval_result, usage = llm_structured_retry(
    client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer is semantically equivalent to the original answer. It confirms that late enrollment is possible but reiterates the condition for receiving a certificate, which is to submit the project while submissions are still open. The key information is identical.', score='good')

In [11]:
calc_price(usage)

{'input_cost': 7.02e-05, 'output_cost': 0.000145, 'total_cost': 0.0002152}

In [16]:
def evaluate_aqa(question, answer_orig, answer_llm, model='gemini-2.5-flash'):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [17]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer is semantically equivalent to the original answer. It confirms that late enrollment is possible but reiterates the condition for receiving a certificate, which is submitting the project within the designated submission window. The key information is identical.', score='good')

In [18]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [19]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/575 [00:00<?, ?it/s]

In [20]:
results[10]

({'question': 'Will students get a Zoom link to join the live office hours?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': "The AI answer directly answers the question by stating that students will not get a Zoom link. It then provides the reason (Zoom link is only for instructors/presenters/TAs) and explains how students participate (YouTube Live and Slido), which aligns perfectly with the original answer's core information. The AI answer correctly captures the key points."},
 GenerateContentResponseUsageMetadata(
   candidates_token_count=81,
   prompt_token_count=325,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=325
     ),
   ],
   thoughts_token_count=105,
   total_token_count=511
 ))

In [21]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [22]:
calc_total_price(usages)

0.20274550000000013

In [23]:
df_eval = pd.DataFrame(evaluations)

In [24]:
df_eval.head()

,question,document,score,reasoning
0,Is it possible to enroll in the course even if...,74eb249bbf,good,The AI answer accurately conveys the same info...
1,What are the requirements for earning a course...,74eb249bbf,bad,The original answer states one requirement for...
2,Is there a specific deadline for submitting th...,74eb249bbf,good,The AI answer correctly identifies that a spec...
3,"If I join the course late, can I still qualify...",74eb249bbf,good,The AI answer correctly states that a student ...
4,Are there any time constraints for project sub...,74eb249bbf,good,The AI answer correctly identifies that there ...


In [25]:
df_eval.score.value_counts()

score
good    536
bad      39
Name: count, dtype: int64

In [26]:
df_eval.score.value_counts(normalize=True)

score
good    0.932174
bad     0.067826
Name: proportion, dtype: float64

In [27]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,What are the requirements for earning a course...,74eb249bbf,bad,The original answer states one requirement for...
27,Do I need to submit homework to get the certif...,69d122f12e,bad,The AI answer correctly states that homework i...
30,What are the requirements to receive a certifi...,9f689c185f,bad,The original answer states that the only requi...
32,What is the main advantage of doing the homewo...,9f689c185f,bad,The user asked for the main advantage of doing...
34,Can I still earn the course certificate if I m...,9f689c185f,bad,The AI answer correctly states that missing ho...


In [29]:
df_eval.to_csv("../data/rag-evaluations-new.csv", index=False)